# ADAPT — SFT Warmup then GRPO Training

**Two-phase training:**
1. **SFT** — Teach the model correct JSON format using heuristic outputs (~15 min)
2. **GRPO** — Optimise decision quality beyond the heuristic (~1 hour)

**Why SFT first?** Without it, ~70% of GRPO episodes produce invalid JSON = wasted compute.

**Recommended runtime:** `Runtime > Change runtime type > GPU (T4)`

## 1. Configuration

In [ ]:
from pathlib import Path
import os

REPO_URL   = "https://github.com/GTsingh600/AirX.git"
REPO_DIR   = Path("/content/ATC")
USE_DRIVE  = True
OUTPUT_DIR = (
    Path("/content/drive/MyDrive/atc-adapt")
    if USE_DRIVE else Path("/content/atc-adapt")
)
SFT_DIR    = str(OUTPUT_DIR / "sft-warmup")
GRPO_DIR   = str(OUTPUT_DIR / "grpo-final")

MODEL_NAME      = "Qwen/Qwen2.5-1.5B-Instruct"
SFT_EPISODES    = 50
SFT_EPOCHS      = 3
GRPO_EPISODES   = 100
LORA_RANK       = 32
N_GENERATIONS   = 2
SEED            = 42

os.environ["WANDB_MODE"]             = "disabled"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONUNBUFFERED"]       = "1"

print(f"MODEL       = {MODEL_NAME}")
print(f"SFT_DIR     = {SFT_DIR}")
print(f"GRPO_DIR    = {GRPO_DIR}")
print(f"SFT:  {SFT_EPISODES} episodes x {SFT_EPOCHS} epochs")
print(f"GRPO: {GRPO_EPISODES} episodes")

## 2. Mount Drive + Clone

In [ ]:
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Output ready: {OUTPUT_DIR}")

In [ ]:
import shutil, subprocess, sys, os
from pathlib import Path

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
print(f"Working dir: {Path.cwd()}")

## 3. Install Dependencies

In [ ]:
import subprocess, sys

def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

pip("-U", "pip")
pip(
    "unsloth[colab-new]",
    "trl==0.15.2",
    "transformers==4.51.3",
    "datasets>=2.20.0",
    "accelerate>=0.32.0",
    "peft>=0.12.0",
    "bitsandbytes>=0.43.0",
    "matplotlib>=3.9.0",
    "numpy>=1.26.0",
    "pydantic>=2.7.0",
)
print("Done.")

## 4. Sanity Check

In [ ]:
import torch
from training.dataset import build_episode_dataset

print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory/1e9:.1f} GB")

sample = build_episode_dataset(
    n_episodes=4, seed=42,
    include_generator=False, include_supervisor=False, include_adapt=True,
    domain_episode_ratio=0.50, long_horizon_ratio=0.0,
)
roles = sorted({r["agent_role"] for r in sample})
print(f"Samples: {len(sample)} | Roles: {roles}")
assert "ADAPT" in roles
assert "GENERATOR" not in roles
print("OK: 3 roles (ADAPT + AMAN + DMAN)")

## 5. Phase 1: SFT Warmup (~15 min)

Teaches JSON format using heuristic outputs. After this, valid JSON rate: ~30% -> ~95%.

In [ ]:
import time
t0 = time.time()

from training.train_sft import train_sft

sft_checkpoint = train_sft(
    model_name  = MODEL_NAME,
    output_dir  = SFT_DIR,
    n_episodes  = SFT_EPISODES,
    lora_rank   = LORA_RANK,
    seed        = SEED,
    num_epochs  = SFT_EPOCHS,
)
print(f"\nSFT done in {(time.time()-t0)/60:.1f} min")
print(f"Checkpoint: {sft_checkpoint}")

## 6. Phase 2: GRPO Training (~1 hour)

Starts from SFT checkpoint. GRPO optimises decision **quality** beyond the heuristic.

You should see: loss decreasing, reward increasing from ~0.3 to ~0.6+

In [ ]:
import time, subprocess, sys, os, re
from collections import defaultdict
from pathlib import Path

os.makedirs(GRPO_DIR, exist_ok=True)

train_cmd = [
    sys.executable, "training/train_grpo.py",
    "--model",         sft_checkpoint,
    "--output_dir",    GRPO_DIR,
    "--episodes",      str(GRPO_EPISODES),
    "--n_generations", str(N_GENERATIONS),
    "--lora_rank",     str(LORA_RANK),
    "--seed",          str(SEED),
]

_ROLES = ("ADAPT", "AMAN", "DMAN", "composite")
reward_history = defaultdict(list)
loss_history = []

_re_atc  = re.compile(r"\[ATC_REWARD step=(\d+)\] (.+)")
_re_kv   = re.compile(r"(\w+)=([\-\d.eE+]+)")
_re_loss = re.compile(r"'loss':\s*([\-\d.eE+]+)")

def _bar(val, width=20):
    frac = max(0.0, min(1.0, val))
    filled = int(frac * width)
    return f"[{'#' * filled}{' ' * (width - filled)}] {val:.3f}"

print("Command:", " ".join(train_cmd))
print("=" * 60)
t0 = time.time()

env_copy = os.environ.copy()
env_copy["PYTHONUNBUFFERED"] = "1"

proc = subprocess.Popen(
    train_cmd,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1, env=env_copy,
)

for raw in proc.stdout:
    line = raw.rstrip("\n")
    print(line, flush=True)

    m_atc = _re_atc.match(line)
    if m_atc:
        for kv in _re_kv.finditer(m_atc.group(2)):
            reward_history[kv.group(1)].append(float(kv.group(2)))

    if "'loss'" in line:
        m_l = _re_loss.search(line)
        if m_l: loss_history.append(float(m_l.group(1)))

proc.wait()
elapsed = time.time() - t0
print("=" * 60)

print(f"\n=== FINAL SUMMARY ===")
for role in _ROLES:
    vals = reward_history.get(role, [])
    if not vals: continue
    avg = sum(vals) / len(vals)
    last = sum(vals[-10:]) / min(10, len(vals))
    print(f"  {role:12s}: avg={avg:.3f}  last10={last:.3f}")

ok = proc.returncode == 0
print(f"\n{'OK' if ok else 'FAILED'} in {elapsed/60:.1f} min")

## 7. Plot Training Curves

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

plots_dir = Path(GRPO_DIR) / "plots"
plots_dir.mkdir(parents=True, exist_ok=True)

COLOURS = {"ADAPT":"#9C27B0", "AMAN":"#2196F3", "DMAN":"#4CAF50", "composite":"#212121"}

def smooth(v, w=20):
    return [sum(v[max(0,i-w):i+1])/min(w,i+1) for i in range(len(v))]

# ── Reward curves ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle("ADAPT SFT+GRPO Reward Curves", fontsize=14, fontweight="bold")

for ax, role in zip(axes.flat, ["ADAPT", "AMAN", "DMAN", "composite"]):
    vals = reward_history.get(role, [])
    if not vals:
        ax.set_title(f"{role} (no data)"); continue
    col = COLOURS.get(role, "#607D8B")
    ax.plot(vals, alpha=0.2, color=col, lw=0.7)
    ax.plot(smooth(vals), color=col, lw=2.0)
    tail = vals[-max(1,len(vals)//4):]
    fm = sum(tail)/len(tail)
    ax.axhline(fm, color=col, lw=1.0, ls=":")
    ax.set_title(f"{role} (final={fm:.3f})", fontweight="bold", color=col)
    ax.set_xlabel("Step")
    ax.set_ylabel("Reward")
    ax.set_ylim(-0.1, 1.05)
    ax.axhline(0, color="gray", lw=0.5, ls="--")
    ax.grid(True, alpha=0.3)

plt.tight_layout()
p1 = plots_dir / "reward_curves.png"
plt.savefig(p1, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved: {p1}")

# ── Loss curve ────────────────────────────────────────────────────────────
if loss_history:
    fig2, ax2 = plt.subplots(figsize=(12, 4))
    ax2.plot(loss_history, color="#F44336", alpha=0.4, lw=0.7)
    ax2.plot(smooth(loss_history, 10), color="#F44336", lw=2.0)
    ax2.set_title("GRPO Loss (should decrease)", fontweight="bold")
    ax2.set_xlabel("Step")
    ax2.set_ylabel("Loss")
    ax2.grid(True, alpha=0.3)
    p2 = plots_dir / "loss_curve.png"
    plt.tight_layout()
    plt.savefig(p2, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {p2}")
    print(f"Loss: start={loss_history[0]:.4f} end={loss_history[-1]:.4f} delta={loss_history[-1]-loss_history[0]:+.4f}")

## Troubleshooting

| Problem | Fix |
|---|---|
| OOM on T4 | Use 1.5B model, LORA_RANK=16 |
| Loss not decreasing | Normal for first few steps. SFT warmup helps. |
| Rewards stuck at 0 | Check SFT checkpoint path. |
| Colab disconnects | Output on Drive. Resume from SFT checkpoint. |